# BoardMind AI: Deep CRNN Handwriting Recognition (HTR)
**Notebook:** `09_handwriting_deep_crnn_training.ipynb`  
**Role:** Senior Computer Vision & Deep Learning Research Engineer  
**Architecture:** Deep Residual CNN (7 Conv Blocks + Skip Connections) + 2-Layer Bidirectional LSTM (512 Hidden Units) + CTC Loss  
**Hardware Target:** Intel Core i9-13900H CPU / CUDA Acceleration

---

## 1. Motivation & Architecture Overview
Classroom whiteboards present challenging conditions: cursive marker strokes, uneven lighting, and variable slant angles.
To achieve high-accuracy handwriting recognition without hallucinating, we implement a **Deep Convolutional Recurrent Neural Network (CRNN)**:

```
Input Image (1, 32, W)
       │
       ▼
[ 1. Deep Residual CNN Backbone (7 Conv Layers + Residual Blocks) ]
• Conv1: 64 filters, 3x3, BatchNorm + LeakyReLU, MaxPool(2, 2)
• Conv2: 128 filters + Residual Identity Block 1
• Conv3: 256 filters + Residual Identity Block 2, MaxPool(2, 1)  <-- Preserves text width
• Conv4: 512 filters + Residual Identity Block 3, MaxPool(2, 1)
• Conv5: 512 filters, 2x1 Conv squash
       │
       ▼  Feature Map: (B, 512, 1, W')
[ 2. Map-to-Sequence Layer ]
• Permutes spatial features into sequential time steps: (W', B, 512)
       │
       ▼
[ 3. Deep Hidden Recurrent Layers (2-Layer BiLSTM) ]
• Hidden Size = 256 (Bidirectional -> 512 units per step)
• Captures forward & backward contextual character flow (n-grams)
• Dropout = 0.3 for regularization
       │
       ▼
[ 4. Connectionist Temporal Classification (CTC) ]
• Projects 512 hidden features to 96 character classes (ASCII + Blank)
• Decodes sequence alignment without requiring per-character bounding box annotations
```

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
import io

# Ensure project root in sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from backend.app.recognition.deep_crnn import create_deep_crnn, DeepCRNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Execution Device: {device}')

## 2. Inspecting the IAM Handwriting Dataset (`datasets/handwriting/`)

In [ ]:
htr_dir = PROJECT_ROOT / 'datasets' / 'handwriting'
train_pq = htr_dir / 'train.parquet'
val_pq = htr_dir / 'val.parquet'

print('Parquet files:', list(htr_dir.glob('*.parquet')))
if train_pq.exists():
    df_sample = pd.read_parquet(train_pq)
    print(f'Total IAM Training Samples: {len(df_sample):,}')
    print('Columns:', df_sample.columns.tolist())
    print('Sample text:', df_sample['text'].head(3).tolist())

## 3. Instantiating the Deep CRNN Model (11.47M Parameters)

In [ ]:
model = create_deep_crnn(num_classes=96).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total DeepCRNN Parameters: {total_params:,}')
print(f'Trainable Parameters: {trainable_params:,}')
print('\nModel Architecture Summary:')
print(model)

## 4. CTC Loss Formulation & Forward Pass Verification

In [ ]:
# Simulate batch of line crops: (Batch=4, Channel=1, Height=32, Width=256)
dummy_input = torch.randn(4, 1, 32, 256).to(device)
log_probs = model(dummy_input)

print(f'Input shape: {dummy_input.shape}')
print(f'Output log-probs shape (Time, Batch, Classes): {log_probs.shape}')

# CTC Loss definition
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
print('CTC Loss initialized with blank_token=0.')

## 5. Training Loop Template with Gradient Clipping & Learning Rate Schedule

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print('Optimizer: AdamW (lr=1e-3, weight_decay=1e-4)')
print('Scheduler: ReduceLROnPlateau')
print('Ready for fine-tuning on IAM handwriting parquet splits.')